# IOAI — 2026 Mock Synthetic Speech (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, urllib.request, gzip, shutil
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-mock-synthetic-speech'
for f in ['X_test.npy','train.csv','test_list.csv']:
    if not os.path.exists(f'data/{f}'): urllib.request.urlretrieve(f'{BASE}/{f}', f'data/{f}')
if not os.path.exists('data/X_train.npy'):
    with open('xt.gz','wb') as out:
        for part in ['Xsp_part_00','Xsp_part_01']:
            urllib.request.urlretrieve(f'{BASE}/{part}', part)
            with open(part,'rb') as p: shutil.copyfileobj(p, out)
    with gzip.open('xt.gz','rb') as g, open('data/X_train.npy','wb') as o: shutil.copyfileobj(g, o)
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Synthetic Speech — 모범답안 (멜 스펙트로그램 CNN)

멜 스펙트로그램을 1채널 이미지로 보고 작은 CNN 을 학습해 spoof/bonafide 를 구분한다(2019 데이터라 비교적
쉬움). ResNet18(사전학습 허용)로도 가능하지만 처음부터 CNN 으로 충분. F1 ≈ 0.9+ (전부 0 인 0 대비). GPU.

In [ ]:
import numpy as np, pandas as pd
Xtr = np.load("data/X_train.npy").astype("float32")   # [N,128,94]
train = pd.read_csv("data/train.csv")                 # filename, label
Xte = np.load("data/X_test.npy").astype("float32")
test_files = pd.read_csv("data/test_list.csv")["filename"].tolist()
print(Xtr.shape, "| label dist", train.label.value_counts().to_dict())

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(0); dev="cuda" if torch.cuda.is_available() else "cpu"
mu, sd = Xtr.mean(), Xtr.std()+1e-6
def norm(X): return ((X-mu)/sd)[:,None,:,:]
Xt=torch.tensor(norm(Xtr)); yt=torch.tensor(train.label.values)
loader=DataLoader(TensorDataset(Xt,yt),batch_size=64,shuffle=True)
class CNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.f=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
                          nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
                          nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),
                          nn.AdaptiveAvgPool2d(1),nn.Flatten())
        s.h=nn.Sequential(nn.Dropout(0.3),nn.Linear(128,2))
    def forward(s,x):return s.h(s.f(x))
net=CNN().to(dev); opt=torch.optim.Adam(net.parameters(),1e-3); lossf=nn.CrossEntropyLoss()
for ep in range(12):
    net.train()
    for xb,yb in loader:
        opt.zero_grad(); loss=lossf(net(xb.to(dev)),yb.to(dev)); loss.backward(); opt.step()
print("trained, loss", round(float(loss),4))
net.eval()
with torch.no_grad(): pred=net(torch.tensor(norm(Xte)).to(dev)).argmax(1).cpu().numpy()
pd.DataFrame({"filename": test_files, "label": pred.astype(int)}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred), "| spoof pred", int(pred.sum()))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)